# 01 — YOLOv8 Detection Training (LOGIVISION)

**Goal:** train a YOLOv8n detector on warehouse frames, log every artefact to MLflow, and register the resulting model in the `logivision-detector` registry.

**What this notebook proves to the jury:**
1. We use a **real dataset** organised in YOLO format (`images/{train,val,test}` + `labels/{train,val,test}`) — see §2.
2. Training is **reproducible**: pinned config, dataset fingerprint hashed into the MLflow tags.
3. We track **every relevant metric** (precision/recall/mAP50/mAP50-95 + per-epoch losses) and the model weights themselves.
4. We can **inspect each run** in MLflow UI (http://localhost:5050) and promote the best one to `Production`.

> The notebook is a thin wrapper around `ml/scripts/train.py` — if you prefer a single command, run `make train` from the repo root.

## 0. Environment

We use `uv` for Python package management (≈10× faster than pip) and Ultralytics for YOLO. The full dependency set is locked in `pyproject.toml`.

In [ ]:
import sys, os, json, hashlib, subprocess
from pathlib import Path

# Make the repo root importable so we can call `ml.scripts.*` as modules.
REPO_ROOT = Path.cwd().resolve()
while not (REPO_ROOT / 'pyproject.toml').is_file() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))
print('Repo root :', REPO_ROOT)
print('Python    :', sys.version.split()[0])

## 1. Data — where it lives, who annotates it

**Source datasets** (pulled by `make fetch-videos` and the Kaggle / TalTech recipes):
* Pexels warehouse stock videos → frames at 2 fps (script: `ml/scripts/extract_frames.py`).
* Kaggle *Warehouse Delivery Box Detection* (when configured) → pre-annotated images.
* TalTech Synthetic Warehouse Videos (when configured).

**Annotation workflow:**
1. Frames are imported into a self-hosted **CVAT** instance (`make cvat-up`, port 8080).
2. Operators draw bounding boxes for: `box`, `pallet`, `forklift`, `person`, `qr_code`, `barcode`.
3. The CVAT job is exported as YOLO 1.1 (zip).
4. `ml/scripts/import_annotations.py` unzips, validates, performs a 70/15/15 train/val/test split, and writes a proper `data.yaml`.

**Versioning:** the entire `datasets/processed/<dataset_version>/` is tracked by **DVC** with MinIO as the remote backend. `dvc.yaml` codifies the pipeline so a fresh clone + `dvc pull && dvc repro` reproduces the exact training set.

In [ ]:
import yaml

DATA_YAML = REPO_ROOT / 'datasets' / 'processed' / 'demo' / 'data.yaml'
assert DATA_YAML.is_file(), f'Run `make demo-data` first — expected {DATA_YAML}'

data = yaml.safe_load(DATA_YAML.read_text())
print('Classes:', data['names'])
print('Train  :', data['train'])
print('Val    :', data['val'])
print('Test   :', data.get('test'))

# Sanity-check: how many label files in each split? An empty split is the
# #1 reason a YOLO run reports precision = 0.
for split in ('train', 'val', 'test'):
    labels_dir = DATA_YAML.parent / 'labels' / split
    n_imgs = len(list((DATA_YAML.parent / 'images' / split).glob('*'))) if (DATA_YAML.parent / 'images' / split).is_dir() else 0
    n_lbls = len(list(labels_dir.glob('*.txt'))) if labels_dir.is_dir() else 0
    print(f'  {split:<5}: {n_imgs:5d} images / {n_lbls:5d} labels')

## 2. Preprocessing — what `extract_frames.py` actually does

Inputs: raw `.mp4` videos in `datasets/raw/videos/`.

For each video:
1. Detect source FPS with OpenCV.
2. `stride = round(source_fps / target_fps)` — sample every `stride`-th frame.
3. If `resize > 0`, scale the longest side to that value (preserves aspect ratio).
4. JPEG-encode at quality 90, write to `datasets/raw/frames/<video_id>/frame_<n>.jpg`.
5. Append one row to `manifest.jsonl` with `(video_id, frame_idx, timestamp_ms, w, h)`.

The manifest is what lets us trace any annotation back to its source video + timestamp — essential for debugging false positives in production.

In [ ]:
# Preview a few sampled frames to confirm the extraction worked.
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

imgs_dir = DATA_YAML.parent / 'images' / 'train'
samples = sorted(imgs_dir.glob('*'))[:6]
fig, axes = plt.subplots(1, len(samples), figsize=(18, 3))
for ax, p in zip(axes, samples):
    ax.imshow(mpimg.imread(p)); ax.set_title(p.name, fontsize=8); ax.axis('off')
plt.suptitle('Training frames (random sample)', y=1.05)
plt.show()

## 3. Training — call the canonical script, NOT a freelance loop

The training logic lives in `ml/scripts/train.py`. We invoke it here so the notebook stays a thin orchestrator — the same script runs in CI, on Colab, and from `make train`.

**Hyperparameters** (`ml/configs/yolov8n.yaml`): 50 epochs, AdamW, lr=0.001, imgsz=640, batch=16, patience=20.

**Why YOLOv8n and not v9/v11?** v8n is the only one with both: (a) license compatible with our AGPL repo, (b) verified OpenVINO export path, (c) <6 MB weights so a CPU laptop hits ≥ 15 FPS after INT8 quantisation.

In [ ]:
# This cell runs a SHORT training (1 epoch) for the notebook demo. For a real
# training run use:    uv run python -m ml.scripts.train --config ml/configs/yolov8n.yaml
import os
os.environ.setdefault('MLFLOW_TRACKING_URI', 'http://localhost:5050')

from ml.scripts.train import train, TrainConfig  # type: ignore[attr-defined]

cfg = TrainConfig(
    config_path=REPO_ROOT / 'ml/configs/yolov8n.yaml',
    overrides={'hyperparameters.epochs': 1},     # quick sanity-check
    dry_register=True,                            # don't pollute the registry
)
result = train(cfg)
print('Run id   :', result.run_id)
print('val_map50:', result.map50)

## 4. Inspecting a real run — what to point at during the defense

The repo ships one completed real run: `under `ml/runs/`` (50 epochs, CPU, AdamW). Its `results.csv` tells the story:

In [ ]:
import pandas as pd

# Discover the most recent completed run rather than hard-coding the ID.
runs_root = REPO_ROOT / 'ml' / 'runs'
candidate = sorted([p for p in runs_root.iterdir() if (p / 'results.csv').is_file()],
                   key=lambda p: p.stat().st_mtime, reverse=True)
assert candidate, 'no completed runs yet — run `make train` first'
run_dir = candidate[0]
print('Inspecting run:', run_dir.name)
df = pd.read_csv(run_dir / 'results.csv')
df = df.rename(columns={c: c.strip() for c in df.columns})  # ultralytics adds spaces
df.tail()

In [ ]:
# Plot the metrics that matter — these are the curves you point at when the jury asks 'did it converge?'
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].plot(df['epoch'], df['metrics/mAP50(B)'], label='mAP50',     color='#2563EB', linewidth=2)
axes[0].plot(df['epoch'], df['metrics/mAP50-95(B)'], label='mAP50-95', color='#06B6D4', linewidth=2)
axes[0].set_xlabel('epoch'); axes[0].set_ylabel('mAP'); axes[0].set_title('Validation mAP'); axes[0].grid(alpha=0.3); axes[0].legend()

axes[1].plot(df['epoch'], df['train/box_loss'], label='box loss', color='#EF4444', linewidth=2)
axes[1].plot(df['epoch'], df['train/cls_loss'], label='cls loss', color='#F59E0B', linewidth=2)
axes[1].plot(df['epoch'], df['train/dfl_loss'], label='dfl loss', color='#8B5CF6', linewidth=2)
axes[1].set_xlabel('epoch'); axes[1].set_ylabel('loss'); axes[1].set_title('Training losses'); axes[1].grid(alpha=0.3); axes[1].legend()
plt.tight_layout(); plt.show()

best = df.loc[df['metrics/mAP50(B)'].idxmax()]
print(f'Best epoch: {int(best.epoch)} → mAP50={best["metrics/mAP50(B)"]:.3f}, '
      f'precision={best["metrics/precision(B)"]:.3f}, recall={best["metrics/recall(B)"]:.3f}')

### Why early epochs show `precision = 0`

If you stare at row 0–3 of `results.csv` you'll see precision values like `0.007`. **This is NOT a broken pipeline** — it's the standard cold-start: a freshly-initialised head outputs near-uniform class scores, so almost every prediction at the default confidence threshold (0.25) is a false positive.

The cure is patience: by epoch 13 precision crosses 0.09 and by epoch 14 it stabilises around 0.12 with recall = 1.0. If you see `precision = 0` after epoch 20 you have a *real* problem — almost always one of:
1. **Empty labels directory** for the split (check `len(list((DATA_YAML.parent / 'labels' / split).glob('*.txt')))`).
2. **Class-id mismatch** between `data.yaml::names` and the label files (the first integer on each line of a `.txt`).
3. **Image extension mismatch** — Ultralytics matches `images/x.jpg` to `labels/x.txt`; if your images are `.png` and labels were exported for `.jpg`, no match.
4. **Single-class collapse** — when `single_cls: true` is set with multi-class labels.

## 5. MLflow integration — what we log, what we tag

`ml/scripts/train.py::train()` does the following inside an `mlflow.start_run()`:

* **Params**: every hyperparameter from `yolov8n.yaml`.
* **Tags**: `git_commit` (from `git rev-parse HEAD`), `dvc_lock_hash` (SHA256 of `data.yaml` + every label file), `model_arch`, `framework`, `dataset_path`.
* **Metrics**: per-epoch losses + final `val_map50`, `val_map50_95`, `val_precision`, `val_recall`.
* **Artifacts**: `best.pt`, `last.pt`, `args.yaml`, confusion matrix PNGs, sample prediction images.
* **Model**: registered as `logivision-detector` at stage `None`. Promotion to `Staging`/`Production` is a separate manual step via `ml/scripts/promote_model.py`.

**Demo:**

In [ ]:
import mlflow
mlflow.set_tracking_uri('http://localhost:5050')
client = mlflow.tracking.MlflowClient()

for exp in client.search_experiments():
    runs = client.search_runs([exp.experiment_id], max_results=5, order_by=['attribute.start_time DESC'])
    if not runs: continue
    print(f'\n=== {exp.name} ===')
    for r in runs:
        m = r.data.metrics
        print(f'  {r.info.run_id[:8]}  status={r.info.status:<8}  '
              f'mAP50={m.get("val_map50", 0):.3f}  precision={m.get("val_precision", 0):.3f}')

## 6. Promotion — turning a run into the live detector

We only promote a run if **all three** thresholds in `ml/configs/promotion_thresholds.yaml` pass:

* `val_map50      ≥ 0.65`
* `val_map50_95   ≥ 0.40`
* `val_recall     ≥ 0.55`

The `8a4db…` run hits mAP50 = 0.865, precision = 0.913, recall = 1.0 → it passes.

```bash
# From the terminal, not the notebook:
make promote          # → Staging
make promote-prod     # → Production (requires --approve flag, see promote_model.py)
```

Once the model is in `Production`, the `inference_worker` picks it up on its next restart (`resolve_model_weights()` queries the registry).

## 7. From this notebook to the live dashboard

1. Notebook calls `ml/scripts/train.py` → MLflow run.
2. `make promote-prod` → MLflow Registry stage = Production.
3. `make inference-worker` (separate terminal) consumes `raw-frames`, runs YOLO, writes to `detections`.
4. `make cep` (separate terminal) consumes `detections`, applies rules, writes to `events`.
5. `make api` (separate terminal) — FastAPI streams `events` via `/ws/events`.
6. The React dashboard's `useEventStream` hook receives every event live → activity feed, collision beacons, KPI tiles update.

**Every step is decoupled by Kafka.** You can kill any one process and the others keep producing/buffering. That's the whole point of Kappa architecture.

## 8. What's next (honest)

* [ ] Train on the Kaggle *Warehouse Delivery Box Detection* dataset (currently using a smaller demo split).
* [ ] Add a held-out test set from TalTech for unbiased mAP reporting.
* [ ] OpenVINO INT8 export (`make export-openvino`) and benchmark in `04_benchmark.ipynb`.
* [ ] Real ByteTrack integration so `track_id` is stable across frames — currently approximated by quantised bbox hash.